# Exercise 3 — Serialization and Deserialization with Marshmallow

Implement the `Stock` / `Trade` serialization contract using **Marshmallow schemas** instead of a hand-written `JSONEncoder` / `JSONDecoder`.

### Goals

- Serialize `Stock` and `Trade` instances to JSON-safe data.
- Deserialize them back into real domain objects.
- Preserve `Decimal` values exactly by representing them as JSON strings.
- Preserve `date` and `datetime` values using ISO-8601.
- Keep explicit `__type__` and `__version__` metadata.
- Reject malformed, unknown, or incompatible payloads.
- Add meaningful domain validation.
- Support nested `activity` data.
- Verify exact round trips with automated assertions.

### Compatibility

The schema code below intentionally uses APIs shared by Marshmallow 3.14.1 and Marshmallow 4.x.

For the original Python 3.6.7 project environment, install:

```bash
pip install marshmallow==3.14.1
```

For a modern Python environment, use a current Marshmallow 4.x release:

```bash
pip install "marshmallow>=4,<5"
```


## 1. Imports and domain model

The classes stay independent from Marshmallow. This keeps the domain model reusable and prevents serialization concerns from leaking into business objects.


In [ ]:
from datetime import date, datetime, timezone
from decimal import Decimal
import json

from marshmallow import (
    RAISE,
    Schema,
    ValidationError,
    fields,
    post_dump,
    post_load,
    pre_load,
    validate,
    validates_schema,
)


SCHEMA_VERSION = 1


In [ ]:
class Stock:
    def __init__(self, symbol, date, open_, high, low, close, volume):
        self.symbol = symbol
        self.date = date
        self.open = open_
        self.high = high
        self.low = low
        self.close = close
        self.volume = volume

    def __repr__(self):
        return (
            "Stock(symbol={!r}, date={!r}, open={!r}, high={!r}, "
            "low={!r}, close={!r}, volume={!r})"
        ).format(
            self.symbol,
            self.date,
            self.open,
            self.high,
            self.low,
            self.close,
            self.volume,
        )

    def __eq__(self, other):
        if not isinstance(other, Stock):
            return NotImplemented

        return (
            self.symbol == other.symbol
            and self.date == other.date
            and self.open == other.open
            and self.high == other.high
            and self.low == other.low
            and self.close == other.close
            and self.volume == other.volume
        )


class Trade:
    def __init__(self, symbol, timestamp, order, price, volume, commission):
        self.symbol = symbol
        self.timestamp = timestamp
        self.order = order
        self.price = price
        self.volume = volume
        self.commission = commission

    def __repr__(self):
        return (
            "Trade(symbol={!r}, timestamp={!r}, order={!r}, price={!r}, "
            "volume={!r}, commission={!r})"
        ).format(
            self.symbol,
            self.timestamp,
            self.order,
            self.price,
            self.volume,
            self.commission,
        )

    def __eq__(self, other):
        if not isinstance(other, Trade):
            return NotImplemented

        return (
            self.symbol == other.symbol
            and self.timestamp == other.timestamp
            and self.order == other.order
            and self.price == other.price
            and self.volume == other.volume
            and self.commission == other.commission
        )


## 2. Sample activity data


In [ ]:
activity = {
    "quotes": [
        Stock(
            "TSLA",
            date(2018, 11, 22),
            Decimal("338.19"),
            Decimal("338.64"),
            Decimal("337.60"),
            Decimal("338.19"),
            365_607,
        ),
        Stock(
            "AAPL",
            date(2018, 11, 22),
            Decimal("176.66"),
            Decimal("177.25"),
            Decimal("176.64"),
            Decimal("176.78"),
            3_699_184,
        ),
        Stock(
            "MSFT",
            date(2018, 11, 22),
            Decimal("103.25"),
            Decimal("103.48"),
            Decimal("103.07"),
            Decimal("103.11"),
            4_493_689,
        ),
    ],
    "trades": [
        Trade(
            "TSLA",
            datetime(2018, 11, 22, 10, 5, 12),
            "buy",
            Decimal("338.25"),
            100,
            Decimal("9.99"),
        ),
        Trade(
            "AAPL",
            datetime(2018, 11, 22, 10, 30, 5),
            "sell",
            Decimal("177.01"),
            20,
            Decimal("9.99"),
        ),
    ],
}

activity


## 3. Reusable tagged-schema base class

Exercises 1 and 2 used explicit `__type__` metadata so serialized objects could be identified safely.

Marshmallow normally knows the target type from the schema itself, but retaining an explicit discriminator has useful properties:

- serialized payloads are self-describing;
- a `Stock` payload cannot accidentally be loaded through `TradeSchema`;
- schema versions can be rejected explicitly;
- no dynamic imports or arbitrary class construction are needed.

The base schema adds tags after dumping and validates/removes them before loading.


In [ ]:
class TaggedSchema(Schema):
    """Base schema for explicitly typed and versioned domain payloads."""

    TYPE_NAME = None
    VERSION = SCHEMA_VERSION

    class Meta:
        unknown = RAISE

    @post_dump
    def add_metadata(self, data, **kwargs):
        result = {
            "__type__": self.TYPE_NAME,
            "__version__": self.VERSION,
        }
        result.update(data)
        return result

    @pre_load
    def validate_and_remove_metadata(self, data, **kwargs):
        if not isinstance(data, dict):
            return data

        payload = dict(data)

        object_type = payload.pop("__type__", None)
        version = payload.pop("__version__", None)

        errors = {}

        if object_type != self.TYPE_NAME:
            errors["__type__"] = [
                "Expected {!r}, got {!r}.".format(
                    self.TYPE_NAME,
                    object_type,
                )
            ]

        if version != self.VERSION:
            errors["__version__"] = [
                "Expected schema version {!r}, got {!r}.".format(
                    self.VERSION,
                    version,
                )
            ]

        if errors:
            raise ValidationError(errors)

        return payload


## 4. Common field factories

`Decimal` values are emitted as strings so the result remains compatible with Python's standard `json` module and never passes through binary floating-point representation.

Factory functions keep monetary-field behavior consistent across schemas.


In [ ]:
def decimal_field(required=True, non_negative=False, positive=False):
    validators = []

    if positive:
        validators.append(
            validate.Range(
                min=Decimal("0"),
                min_inclusive=False,
            )
        )
    elif non_negative:
        validators.append(
            validate.Range(
                min=Decimal("0"),
            )
        )

    return fields.Decimal(
        required=required,
        as_string=True,
        allow_nan=False,
        validate=validators or None,
    )


def symbol_field():
    return fields.String(
        required=True,
        validate=validate.Length(min=1, max=15),
    )


## 5. `StockSchema`

Besides type conversion, the schema validates basic OHLC consistency:

- `low <= high`
- `low <= open <= high`
- `low <= close <= high`
- volume cannot be negative

`@post_load` is the bridge from a validated dictionary to a real `Stock` instance.


In [ ]:
class StockSchema(TaggedSchema):
    TYPE_NAME = "Stock"

    symbol = symbol_field()
    date = fields.Date(required=True, format="iso")

    open = decimal_field()
    high = decimal_field()
    low = decimal_field()
    close = decimal_field()

    volume = fields.Integer(
        required=True,
        strict=True,
        validate=validate.Range(min=0),
    )

    @validates_schema
    def validate_ohlc(self, data, **kwargs):
        low = data["low"]
        high = data["high"]
        open_ = data["open"]
        close = data["close"]

        errors = {}

        if low > high:
            errors.setdefault("low", []).append(
                "low cannot be greater than high."
            )

        if not low <= open_ <= high:
            errors.setdefault("open", []).append(
                "open must be between low and high."
            )

        if not low <= close <= high:
            errors.setdefault("close", []).append(
                "close must be between low and high."
            )

        if errors:
            raise ValidationError(errors)

    @post_load
    def make_stock(self, data, **kwargs):
        return Stock(
            symbol=data["symbol"],
            date=data["date"],
            open_=data["open"],
            high=data["high"],
            low=data["low"],
            close=data["close"],
            volume=data["volume"],
        )


## 6. `TradeSchema`

The trade schema validates:

- `order` is either `buy` or `sell`;
- `price` is positive;
- `volume` is a positive integer;
- `commission` is non-negative.


In [ ]:
class TradeSchema(TaggedSchema):
    TYPE_NAME = "Trade"

    symbol = symbol_field()

    timestamp = fields.DateTime(
        required=True,
        format="iso",
    )

    order = fields.String(
        required=True,
        validate=validate.OneOf(("buy", "sell")),
    )

    price = decimal_field(positive=True)

    volume = fields.Integer(
        required=True,
        strict=True,
        validate=validate.Range(
            min=0,
            min_inclusive=False,
        ),
    )

    commission = decimal_field(non_negative=True)

    @post_load
    def make_trade(self, data, **kwargs):
        return Trade(
            symbol=data["symbol"],
            timestamp=data["timestamp"],
            order=data["order"],
            price=data["price"],
            volume=data["volume"],
            commission=data["commission"],
        )


## 7. `ActivitySchema`

The outer dictionary has a known structure, so `fields.Nested` provides typed recursive serialization/deserialization without a custom JSON object hook.


In [ ]:
class ActivitySchema(Schema):
    quotes = fields.List(
        fields.Nested(StockSchema),
        required=True,
    )

    trades = fields.List(
        fields.Nested(TradeSchema),
        required=True,
    )

    class Meta:
        unknown = RAISE


stock_schema = StockSchema()
trade_schema = TradeSchema()
activity_schema = ActivitySchema()


## 8. Serialize one object

Marshmallow's `dump()` returns JSON-safe Python data. The explicit type/version metadata is produced by `TaggedSchema`.


In [ ]:
stock = activity["quotes"][0]

stock_data = stock_schema.dump(stock)

print(stock_data)
print()
print(json.dumps(stock_data, indent=2, sort_keys=True))


## 9. Deserialize one object


In [ ]:
restored_stock = stock_schema.load(stock_data)

assert isinstance(restored_stock, Stock)
assert restored_stock == stock
assert isinstance(restored_stock.date, date)
assert isinstance(restored_stock.open, Decimal)

print(restored_stock)
print("Single Stock round-trip passed.")


## 10. Serialize the complete activity structure


In [ ]:
serialized_activity = activity_schema.dumps(
    activity,
    indent=2,
    sort_keys=True,
)

print(serialized_activity)


## 11. Deserialize the complete activity structure


In [ ]:
decoded_activity = activity_schema.loads(serialized_activity)

print(decoded_activity)


## 12. End-to-end round-trip verification


In [ ]:
assert decoded_activity == activity

assert all(
    isinstance(item, Stock)
    for item in decoded_activity["quotes"]
)

assert all(
    isinstance(item.date, date)
    and not isinstance(item.date, datetime)
    for item in decoded_activity["quotes"]
)

assert all(
    isinstance(value, Decimal)
    for item in decoded_activity["quotes"]
    for value in (
        item.open,
        item.high,
        item.low,
        item.close,
    )
)

assert all(
    isinstance(item, Trade)
    for item in decoded_activity["trades"]
)

assert all(
    isinstance(item.timestamp, datetime)
    for item in decoded_activity["trades"]
)

assert all(
    isinstance(item.price, Decimal)
    and isinstance(item.commission, Decimal)
    for item in decoded_activity["trades"]
)

print("Complete activity round-trip passed.")


## 13. Exact decimal precision

This test demonstrates why `Decimal(as_string=True)` is important for financial data.


In [ ]:
precision_trade = Trade(
    symbol="TEST",
    timestamp=datetime(2024, 5, 17, 14, 22, 31, 987654),
    order="buy",
    price=Decimal("1234567890.12345678901234567890"),
    volume=1,
    commission=Decimal("0.00000000000000000001"),
)

precision_json = trade_schema.dumps(precision_trade)
precision_result = trade_schema.loads(precision_json)

assert precision_result == precision_trade
assert precision_result.price == Decimal(
    "1234567890.12345678901234567890"
)
assert precision_result.commission == Decimal(
    "0.00000000000000000001"
)

print(precision_json)
print()
print("Exact Decimal precision preserved.")


## 14. Timezone-aware datetime round trip

ISO serialization should preserve timezone information as well as microseconds.


In [ ]:
aware_trade = Trade(
    symbol="TEST",
    timestamp=datetime(
        2024,
        5,
        17,
        14,
        22,
        31,
        123456,
        tzinfo=timezone.utc,
    ),
    order="sell",
    price=Decimal("42.50"),
    volume=2,
    commission=Decimal("0"),
)

aware_json = trade_schema.dumps(aware_trade)
aware_result = trade_schema.loads(aware_json)

assert aware_result == aware_trade
assert aware_result.timestamp.tzinfo is not None
assert aware_result.timestamp.microsecond == 123456

print(aware_json)
print()
print("Timezone-aware datetime round-trip passed.")


## 15. Validation: incorrect type discriminator

A `Trade` payload must not be accepted by `StockSchema`, even if some fields happen to overlap.


In [ ]:
wrong_type_payload = {
    "__type__": "Trade",
    "__version__": SCHEMA_VERSION,
    "symbol": "TSLA",
    "date": "2018-11-22",
    "open": "338.19",
    "high": "338.64",
    "low": "337.60",
    "close": "338.19",
    "volume": 365607,
}

try:
    stock_schema.load(wrong_type_payload)
except ValidationError as exc:
    print("Incorrect type rejected:")
    print(exc.messages)
else:
    raise AssertionError("Expected ValidationError")


## 16. Validation: incompatible schema version


In [ ]:
wrong_version_payload = {
    "__type__": "Stock",
    "__version__": 999,
    "symbol": "TSLA",
    "date": "2018-11-22",
    "open": "338.19",
    "high": "338.64",
    "low": "337.60",
    "close": "338.19",
    "volume": 365607,
}

try:
    stock_schema.load(wrong_version_payload)
except ValidationError as exc:
    print("Unsupported version rejected:")
    print(exc.messages)
else:
    raise AssertionError("Expected ValidationError")


## 17. Validation: unknown fields

Unknown fields are rejected explicitly rather than silently discarded.


In [ ]:
unknown_field_payload = {
    "__type__": "Trade",
    "__version__": SCHEMA_VERSION,
    "symbol": "TSLA",
    "timestamp": "2018-11-22T10:05:12",
    "order": "buy",
    "price": "338.25",
    "volume": 100,
    "commission": "9.99",
    "unexpected": "should fail",
}

try:
    trade_schema.load(unknown_field_payload)
except ValidationError as exc:
    print("Unknown field rejected:")
    print(exc.messages)
else:
    raise AssertionError("Expected ValidationError")


## 18. Validation: invalid trade values


In [ ]:
invalid_trade_payload = {
    "__type__": "Trade",
    "__version__": SCHEMA_VERSION,
    "symbol": "TSLA",
    "timestamp": "2018-11-22T10:05:12",
    "order": "hold",
    "price": "-1.00",
    "volume": 0,
    "commission": "-0.01",
}

try:
    trade_schema.load(invalid_trade_payload)
except ValidationError as exc:
    print("Invalid trade rejected:")
    print(json.dumps(exc.messages, indent=2, sort_keys=True))
else:
    raise AssertionError("Expected ValidationError")


## 19. Validation: inconsistent OHLC values


In [ ]:
invalid_stock_payload = {
    "__type__": "Stock",
    "__version__": SCHEMA_VERSION,
    "symbol": "TSLA",
    "date": "2018-11-22",
    "open": "340.00",
    "high": "338.64",
    "low": "337.60",
    "close": "336.00",
    "volume": 365607,
}

try:
    stock_schema.load(invalid_stock_payload)
except ValidationError as exc:
    print("Invalid OHLC payload rejected:")
    print(json.dumps(exc.messages, indent=2, sort_keys=True))
else:
    raise AssertionError("Expected ValidationError")


## 20. Application-facing API

Business code should depend on a small serialization interface rather than repeatedly configuring schemas.


In [ ]:
def dumps_activity(value, **json_kwargs):
    """Serialize an activity dictionary to JSON."""
    return activity_schema.dumps(value, **json_kwargs)


def loads_activity(payload):
    """Deserialize and validate activity JSON."""
    if not isinstance(payload, str):
        raise TypeError(
            "payload must be str, got {}".format(
                type(payload).__name__
            )
        )

    return activity_schema.loads(payload)


def round_trip(value):
    """Convenience helper used by tests and examples."""
    return loads_activity(
        dumps_activity(
            value,
            sort_keys=True,
            separators=(",", ":"),
        )
    )


In [ ]:
result = round_trip(activity)

assert result == activity
assert result is not activity
assert result["quotes"][0] is not activity["quotes"][0]
assert result["trades"][0] is not activity["trades"][0]

print("=" * 60)
print("ALL EXERCISE 3 TESTS PASSED")
print("=" * 60)


## 21. What Marshmallow replaced

Compared with Exercises 1 and 2, Marshmallow now owns most of the serialization protocol:

| Concern | Custom JSON approach | Marshmallow approach |
|---|---|---|
| Field conversion | `JSONEncoder.default()` | declared `fields.*` |
| Reconstruction | `object_hook` branches | `@post_load` |
| Nested objects | recursive JSON hooks | `fields.Nested` |
| Decimal handling | manual tag + string | `fields.Decimal(as_string=True)` |
| Date/time handling | manual ISO conversion/parsing | `fields.Date` / `fields.DateTime` |
| Input validation | handwritten checks | field + schema validators |
| Unknown fields | custom handling | `unknown = RAISE` |
| Type/version metadata | handwritten | reusable `TaggedSchema` hooks |

The important architectural difference is that deserialization is now **schema-driven**. There is no dynamic class lookup from untrusted JSON.

This completes **Exercise 3**.
